## Parte 1 — Softmax con temperatura

La softmax con temperatura se define como p_i = exp(z_i/T) / Σ_j exp(z_j/T). Para implementarla de forma numéricamente estable se resta el máximo de los logits m antes de exponenciar, p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), que es matemáticamente idéntica pero evita desbordamientos. Aplicada a los logits [2.0, 1.0, 0.5, 0.2, −1.0, −3.0], las tres distribuciones suman 1.0, lo que verifica la implementación:

| token | T=0.5 | T=1 | T=2 |
|---|---|---|---|
| t0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.0225 | 0.0912 | 0.1457 |
| t4 | 0.0020 | 0.0275 | 0.0800 |
| t5 | 0.0000 | 0.0037 | 0.0294 |
| **entropía (bits)** | **0.9147** | **1.7998** | **2.2888** |

El 0.0000 de t5 con T=0.5 corresponde a 3.7367e-05 antes de redondear. La temperatura actúa como se espera: al bajarla (T=0.5) la masa se concentra en el token de mayor logit (t0 sube de 0.5516 a 0.8231 y t5 cae a 3.7367e-05); al subirla (T=2) la distribución se aplana (t5 sube a 0.0294 y t0 baja a 0.3583). La entropía en bits crece monótonamente con T: 0.9147 → 1.7998 → 2.2888.

In [1]:
# T1: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
SUBTAREA T1 — Softmax con temperatura (NumPy, numéricamente estable) y entropía en bits.
Tarea B — Temperatura, top-p y entropía de la distribución de salida (MMIA 6013).

Salidas:
  - resultados.json            (contrato: distribuciones, entropías, tabla a 4 decimales)
  - t1_softmax_temperatura.png (figura: distribuciones y entropía vs. temperatura)
"""

import json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 1. Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]
etiquetas = ["T=0.5", "T=1", "T=2"]


# ----------------------------------------------------------------------
# 2. Softmax con temperatura, numéricamente estable
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    """
    p_i = exp(z_i / T) / sum_j exp(z_j / T), implementación estable.

    Restar una constante m a todos los logits no cambia la softmax:
        exp((z_i - m)/T) / sum_j exp((z_j - m)/T) == exp(z_i/T) / sum_j exp(z_j/T)
    Tomando m = max(z) se evita el overflow de exp para logits grandes.
    """
    z = np.asarray(z, dtype=float)
    m = np.max(z)                      # constante de estabilidad numérica
    e = np.exp((z - m) / T)
    return e / e.sum()


def entropia_bits(p):
    """Entropía de Shannon en bits: H(p) = -sum_i p_i * log2(p_i), con 0*log2(0)=0."""
    p = np.asarray(p, dtype=float)
    p_pos = p[p > 0]
    return float(-np.sum(p_pos * np.log2(p_pos)))


# ----------------------------------------------------------------------
# 3. Cálculo de las tres distribuciones y sus entropías
# ----------------------------------------------------------------------
distribuciones = {lab: softmax_con_temperatura(logits, T)
                  for lab, T in zip(etiquetas, temperaturas)}
entropias = {lab: entropia_bits(distribuciones[lab]) for lab in etiquetas}
sumas = {lab: float(distribuciones[lab].sum()) for lab in etiquetas}

# ----------------------------------------------------------------------
# 4. Tabla con cuatro decimales (distribuciones + entropías)
# ----------------------------------------------------------------------
def f4(v):
    return f"{v:.4f}"

tabla = pd.DataFrame(
    {lab: [f4(v) for v in distribuciones[lab]] for lab in etiquetas},
    index=tokens,
)
tabla.loc["entropia_bits"] = [f4(entropias[lab]) for lab in etiquetas]

# ----------------------------------------------------------------------
# 5. Figura PNG
# ----------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

x = np.arange(len(tokens))
ancho = 0.26
colores = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for k, lab in enumerate(etiquetas):
    ax1.bar(x + (k - 1) * ancho, distribuciones[lab], width=ancho,
            label=f"{lab} (H = {entropias[lab]:.4f} bits)", color=colores[k])
ax1.set_xticks(x)
ax1.set_xticklabels(tokens)
ax1.set_xlabel("Token")
ax1.set_ylabel("Probabilidad")
ax1.set_title("Softmax con temperatura")
ax1.legend(fontsize=8)

ax2.plot(temperaturas, [entropias[lab] for lab in etiquetas], "o-", color="#d62728")
for T, lab in zip(temperaturas, etiquetas):
    ax2.annotate(f"{entropias[lab]:.4f}", (T, entropias[lab]),
                 textcoords="offset points", xytext=(6, 4), fontsize=9)
ax2.set_xticks(temperaturas)
ax2.set_xlabel("Temperatura T")
ax2.set_ylabel("Entropía (bits)")
ax2.set_title("Entropía vs. temperatura")
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 6. resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "subtarea": "T1_softmax_con_temperatura",
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {lab: distribuciones[lab].tolist() for lab in etiquetas},
    "entropias_bits": {lab: entropias[lab] for lab in etiquetas},
    "suma_de_cada_distribucion": sumas,
    "tabla_cuatro_decimales": {
        "columnas": ["token"] + etiquetas,
        "filas": (
            [[tok] + [tabla.loc[tok, lab] for lab in etiquetas] for tok in tokens]
            + [["entropia_bits"] + [tabla.loc["entropia_bits", lab] for lab in etiquetas]]
        ),
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# 7. Resumen
# ----------------------------------------------------------------------
print("SUBTAREA T1 — Softmax con temperatura y entropía en bits")
print(f"Logits: {logits.tolist()}")
print()
print(tabla.to_string())
print()
for lab in etiquetas:
    print(f"{lab}: suma de probabilidades = {sumas[lab]:.12f} | "
          f"entropía = {entropias[lab]:.4f} bits")
print()
print("Archivos escritos: resultados.json, t1_softmax_temperatura.png")


SUBTAREA T1 — Softmax con temperatura y entropía en bits
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

                T=0.5     T=1     T=2
t0             0.8231  0.5516  0.3583
t1             0.1114  0.2029  0.2173
t2             0.0410  0.1231  0.1693
t3             0.0225  0.0912  0.1457
t4             0.0020  0.0275  0.0800
t5             0.0000  0.0037  0.0294
entropia_bits  0.9147  1.7998  2.2888

T=0.5: suma de probabilidades = 1.000000000000 | entropía = 0.9147 bits
T=1: suma de probabilidades = 1.000000000000 | entropía = 1.7998 bits
T=2: suma de probabilidades = 1.000000000000 | entropía = 2.2888 bits

Archivos escritos: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T=1 de la Parte 1 se ordenan los tokens por probabilidad descendente y se acumula hasta alcanzar p=0.9. Tras t0, t1 y t2 la probabilidad acumulada es 0.8776, aún por debajo de 0.9, por lo que debe incluirse t3, con el que la acumulada llega a 0.9688 ≥ 0.9. El núcleo mínimo es {t0, t1, t2, t3} (4 tokens) y quedan filtrados t4 y t5. La renormalización divide cada probabilidad superviviente entre la masa del núcleo, 0.9688, lo que eleva ligeramente todas las probabilidades (p. ej., t0 pasa de 0.5516 a 0.5694). La distribución renormalizada suma 1.0.

| token | prob. (T=1) | prob. acumulada | ¿superviviente? | prob. renormalizada |
|---|---|---|---|---|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | — |
| t5 | 0.0037 | 1.0000 | no | — |

In [2]:
# T2: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =====================================================================
# T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución T=1
# =====================================================================

P = 0.9


def softmax_T1(logits):
    """Softmax con temperatura T=1, numéricamente estable (resta del máximo)."""
    z = np.asarray(logits, dtype=float)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


# ---------------------------------------------------------------------
# 1) Distribución de T=1 de la Parte 1 (entrada/T1/resultados.json)
# ---------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
p_t1 = softmax_T1(logits)
fuente = "respaldo: softmax estable recalculada con T=1"

ruta_t1 = Path("entrada/T1/resultados.json")
if ruta_t1.exists():
    try:
        with ruta_t1.open("r", encoding="utf-8") as f:
            res_t1 = json.load(f)
        tokens = list(res_t1["tokens"])
        logits = np.asarray(res_t1["logits"], dtype=float)
        p_t1 = np.asarray(res_t1["distribuciones"]["T=1"], dtype=float)
        fuente = "entrada/T1/resultados.json (distribución de T=1 de la Parte 1)"
        # Consistencia: lo cargado coincide con la softmax T=1 bien calculada
        assert np.allclose(p_t1, softmax_T1(logits), atol=1e-12)
    except (KeyError, TypeError, AssertionError):
        tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
        logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
        p_t1 = softmax_T1(logits)
        fuente = "respaldo: softmax estable recalculada con T=1"

# ---------------------------------------------------------------------
# 2) Ordenar tokens por probabilidad descendente y acumular
# ---------------------------------------------------------------------
orden = np.argsort(-p_t1, kind="stable")
tokens_ord = [tokens[i] for i in orden]
probs_ord = p_t1[orden]
acum_ord = np.cumsum(probs_ord)

# ---------------------------------------------------------------------
# 3) Conjunto MÍNIMO cuya probabilidad acumulada alcanza p = 0.9
#    (se incluye el token que hace cruzar el umbral)
# ---------------------------------------------------------------------
n_nucleo = int(np.searchsorted(acum_ord, P, side="left")) + 1
idx_nucleo = orden[:n_nucleo]
tokens_nucleo = [tokens[i] for i in idx_nucleo]
tokens_fuera = [tokens[i] for i in orden[n_nucleo:]]
acum_nucleo = float(acum_ord[n_nucleo - 1])

# ---------------------------------------------------------------------
# 4) Renormalizar las probabilidades de los supervivientes
# ---------------------------------------------------------------------
p_nucleo = p_t1[idx_nucleo]
p_renorm = p_nucleo / p_nucleo.sum()
suma_renorm = float(p_renorm.sum())
dist_renorm_dict = {tokens[i]: float(p_renorm[j]) for j, i in enumerate(idx_nucleo)}

# ---------------------------------------------------------------------
# 5) Verificaciones del criterio de éxito
# ---------------------------------------------------------------------
assert abs(suma_renorm - 1.0) < 1e-12, "La distribución renormalizada debe sumar 1"
assert acum_nucleo >= P, "La probabilidad acumulada del núcleo debe alcanzar p"
if n_nucleo > 1:
    assert acum_ord[n_nucleo - 2] < P, "El núcleo debe ser el conjunto mínimo"
assert np.allclose(p_renorm, p_t1[idx_nucleo] / acum_nucleo, atol=0)

# ---------------------------------------------------------------------
# 6) Tabla (4 decimales) con el procedimiento completo
# ---------------------------------------------------------------------
p_renorm_full = np.concatenate([p_renorm, np.zeros(len(tokens_ord) - n_nucleo)])
tabla_filas = []
for j, tok in enumerate(tokens_ord):
    es_nuc = j < n_nucleo
    tabla_filas.append([
        tok,
        f"{probs_ord[j]:.4f}",
        f"{acum_ord[j]:.4f}",
        "sí" if es_nuc else "no",
        f"{p_renorm_full[j]:.4f}" if es_nuc else None,
    ])

# ---------------------------------------------------------------------
# 7) Figura: distribución original vs renormalizada y acumulada
# ---------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
x = np.arange(len(tokens_ord))
colores = ["#2a9d8f" if j < n_nucleo else "#d95f02" for j in range(len(tokens_ord))]

ax1.bar(x - 0.2, probs_ord, width=0.4, color=colores, label="T=1 original")
ax1.bar(x + 0.2, p_renorm_full, width=0.4, color=colores, alpha=0.45,
        label="top-p renormalizada")
ax1.set_xticks(x)
ax1.set_xticklabels(tokens_ord)
ax1.set_ylabel("probabilidad")
ax1.set_title(f"Top-p (p={P}): núcleo = {{{', '.join(tokens_nucleo)}}}")
ax1.legend()

ax2.plot(np.arange(1, len(tokens_ord) + 1), acum_ord, marker="o",
         color="#457b9d", label="prob. acumulada (orden desc.)")
ax2.axhline(P, color="#d95f02", ls="--", label=f"p = {P}")
ax2.axvline(n_nucleo, color="gray", ls=":", label="corte del núcleo")
ax2.annotate(f"acum. núcleo = {acum_nucleo:.4f}",
             xy=(n_nucleo, acum_nucleo), xytext=(n_nucleo + 0.2, acum_nucleo - 0.12),
             fontsize=9)
ax2.set_xticks(np.arange(1, len(tokens_ord) + 1))
ax2.set_xticklabels(tokens_ord)
ax2.set_ylabel("probabilidad acumulada")
ax2.set_title("Acumulado y corte top-p")
ax2.legend(loc="lower right")

fig.tight_layout()
fig.savefig("T2_top_p_nucleus.png", dpi=120)
plt.close(fig)

# ---------------------------------------------------------------------
# 8) resultados.json (contrato de la subtarea)
# ---------------------------------------------------------------------
resultados = {
    "subtarea": "T2_top_p_nucleus_sampling",
    "p": P,
    "temperatura": 1.0,
    "fuente_distribucion": fuente,
    "tokens": tokens,
    "logits": logits.tolist(),
    "distribucion_T1_original": p_t1.tolist(),
    "suma_distribucion_T1_original": float(p_t1.sum()),
    "orden_descendente": tokens_ord,
    "probs_ordenadas": probs_ord.tolist(),
    "prob_acumulada_ordenada": acum_ord.tolist(),
    "tokens_sobrevivientes": tokens_nucleo,
    "tokens_filtrados": tokens_fuera,
    "num_tokens_nucleo": n_nucleo,
    "prob_acumulada_nucleo": acum_nucleo,
    "distribucion_renormalizada": dist_renorm_dict,
    "distribucion_renormalizada_orden_descendente": p_renorm.tolist(),
    "suma_distribucion_renormalizada": suma_renorm,
    "tabla_cuatro_decimales": {
        "columnas": ["token", "prob_T1_original", "prob_acumulada",
                     "superviviente", "prob_renormalizada"],
        "filas": tabla_filas,
    },
    "figura": "T2_top_p_nucleus.png",
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------------------------
# 9) Resumen
# ---------------------------------------------------------------------
print("T2 — Top-p (nucleus sampling) con p=0.9 sobre la distribución T=1 de la Parte 1")
print(f"Fuente de la distribución: {fuente}")
print(f"Orden descendente: {tokens_ord}")
print(f"Prob. acumulada (orden desc.): {[round(a, 4) for a in acum_ord.tolist()]}")
print(f"Tokens supervivientes ({n_nucleo}): {tokens_nucleo} "
      f"| prob. acumulada del núcleo = {acum_nucleo:.6f}")
print(f"Tokens filtrados: {tokens_fuera}")
print("Distribución renormalizada:")
for t, v in dist_renorm_dict.items():
    print(f"  {t}: {v:.6f}")
print(f"Suma de la distribución renormalizada: {suma_renorm:.12f}")
print("Figura: T2_top_p_nucleus.png | Resultados: resultados.json")


T2 — Top-p (nucleus sampling) con p=0.9 sobre la distribución T=1 de la Parte 1
Fuente de la distribución: entrada/T1/resultados.json (distribución de T=1 de la Parte 1)
Orden descendente: ['t0', 't1', 't2', 't3', 't4', 't5']
Prob. acumulada (orden desc.): [0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0]
Tokens supervivientes (4): ['t0', 't1', 't2', 't3'] | prob. acumulada del núcleo = 0.968820
Tokens filtrados: ['t4', 't5']
Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
Suma de la distribución renormalizada: 1.000000000000
Figura: T2_top_p_nucleus.png | Resultados: resultados.json


## Parte 3 — Comprobación empírica

Se tomaron 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Las frecuencias observadas replican de cerca las probabilidades teóricas: las diferencias obs − teo son −0.0012, −0.0002, +0.0026 y −0.0012 para t0–t3, con una desviación absoluta máxima de +0.0026 (t2). La divergencia KL de la distribución empírica respecto de la teórica es 5.0069e-05 bits, prácticamente cero, del orden esperable por pura fluctuación de muestreo con n=10 000. El gráfico de barras generado en la celda muestra, para cada token, las barras observadas casi indistinguibles de las teóricas.

| token | prob. teórica | conteo observado | frecuencia observada | dif. (obs − teo) |
|---|---|---|---|---|
| t0 | 0.5694 | 5682 | 0.5682 | −0.0012 |
| t1 | 0.2095 | 2093 | 0.2093 | −0.0002 |
| t2 | 0.1270 | 1296 | 0.1296 | +0.0026 |
| t3 | 0.0941 | 929 | 0.0929 | −0.0012 |

In [3]:
# T3: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# T3: Comprobación empírica de la distribución renormalizada (top-p, p=0.9)
# ---------------------------------------------------------------

# 1) Cargar la distribución renormalizada de la Parte 2 (subtarea T2)
ruta_t2 = Path("entrada/T2/resultados.json")
with open(ruta_t2, "r", encoding="utf-8") as f:
    t2 = json.load(f)

tokens = t2["tokens_sobrevivientes"]          # ["t0", "t1", "t2", "t3"]
dist_renorm = t2["distribucion_renormalizada"]

# Probabilidades teóricas (distribución renormalizada) en orden de tokens
p_teorica = np.array([dist_renorm[t] for t in tokens], dtype=float)
assert abs(p_teorica.sum() - 1.0) < 1e-12, "La distribución teórica debe sumar 1"

# 2) Tomar 10,000 muestras con numpy.random.default_rng(0)
SEMILLA = 0
N_MUESTRAS = 10_000
rng = np.random.default_rng(SEMILLA)
indices = rng.choice(len(tokens), size=N_MUESTRAS, p=p_teorica)

# 3) Distribución empírica (frecuencias observadas)
conteos = np.bincount(indices, minlength=len(tokens))
f_obs = conteos / N_MUESTRAS

# 4) Divergencia KL (empírica || teórica) en bits
#    D_KL(P||Q) = sum_i P_i * log2(P_i / Q_i); términos con P_i = 0 valen 0
mask = f_obs > 0
kl_bits = float(np.sum(f_obs[mask] * np.log2(f_obs[mask] / p_teorica[mask])))

# 5) Diagrama de barras: frecuencias observadas vs probabilidades teóricas
x = np.arange(len(tokens))
ancho = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - ancho / 2, f_obs, ancho, color="#4C72B0",
            edgecolor="black", label="Frecuencia observada (empírica)")
b2 = ax.bar(x + ancho / 2, p_teorica, ancho, color="#DD8452",
            edgecolor="black", label="Probabilidad teórica (renormalizada)")
for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.006,
                f"{b.get_height():.4f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / Frecuencia relativa")
ax.set_title("Parte 3 — Muestreo top-p (p=0.9): 10,000 muestras con default_rng(0)\n"
             f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
ax.set_ylim(0, max(f_obs.max(), p_teorica.max()) * 1.18)
ax.legend()
plt.tight_layout()
plt.savefig("T3_comparacion_frecuencias_kl.png", dpi=120)
plt.close(fig)

# 6) Escribir resultados.json (contrato de la subtarea)
resultados = {
    "subtarea": "T3_comprobacion_empirica_kl",
    "n_muestras": N_MUESTRAS,
    "semilla": SEMILLA,
    "generador": "numpy.random.default_rng(0)",
    "fuente_distribucion": "entrada/T2/resultados.json (distribución renormalizada top-p p=0.9 de la Parte 2)",
    "tokens_sobrevivientes": tokens,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens, p_teorica)},
    "probabilidades_teoricas_lista": [float(p) for p in p_teorica],
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(fr) for t, fr in zip(tokens, f_obs)},
    "frecuencias_observadas_lista": [float(fr) for fr in f_obs],
    "suma_frecuencias_observadas": float(f_obs.sum()),
    "divergencia_KL_bits": kl_bits,
    "tabla_comparacion": {
        "columnas": ["token", "prob_teorica", "conteo_observado",
                     "frecuencia_observada", "diferencia_obs-teo"],
        "filas": [
            [t, f"{p:.4f}", int(c), f"{fr:.4f}", f"{fr - p:+.4f}"]
            for t, p, c, fr in zip(tokens, p_teorica, conteos, f_obs)
        ],
    },
    "figura": "T3_comparacion_frecuencias_kl.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# 7) Resumen breve
print("=== T3: Comprobación empírica (top-p p=0.9) ===")
print(f"Muestras: {N_MUESTRAS} con numpy.random.default_rng({SEMILLA})")
print(f"Tokens supervivientes: {tokens}")
for t, p, c, fr in zip(tokens, p_teorica, conteos, f_obs):
    print(f"  {t}: teórica={p:.4f} | observada={fr:.4f} (conteo={c})")
print(f"Divergencia KL (empírica || teórica) = {kl_bits:.6f} bits")
print("Figura guardada: T3_comparacion_frecuencias_kl.png")
print("Resultados escritos en resultados.json")


=== T3: Comprobación empírica (top-p p=0.9) ===
Muestras: 10000 con numpy.random.default_rng(0)
Tokens supervivientes: ['t0', 't1', 't2', 't3']
  t0: teórica=0.5694 | observada=0.5682 (conteo=5682)
  t1: teórica=0.2095 | observada=0.2093 (conteo=2093)
  t2: teórica=0.1270 | observada=0.1296 (conteo=1296)
  t3: teórica=0.0941 | observada=0.0929 (conteo=929)
Divergencia KL (empírica || teórica) = 0.000050 bits
Figura guardada: T3_comparacion_frecuencias_kl.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

Cuando T → 0, el cociente z_i/T amplifica las diferencias entre logits: exp((z_i − m)/T) tiende a 0 para todo token cuyo logit sea menor que el máximo, y toda la masa se concentra en el token de mayor logit (aquí t0). La distribución tiende a una delta (one-hot) de entropía 0. Cuando T → ∞, z_i/T → 0 para todos los tokens, exp(z_i/T) → 1 y p_i → 1/6: la distribución tiende a la uniforme sobre el vocabulario, cuya entropía es log2(6), el máximo posible para seis tokens.

T → 0 equivale a la decodificación voraz porque greedy selecciona argmax_i z_i, y la softmax con T → 0 asigna probabilidad 1 precisamente a ese token: muestrear de esa distribución devuelve casi seguramente el mismo token que elegiría greedy.

Las entropías de la Parte 1 son la evidencia empírica de ambos límites: 0.9147 bits con T=0.5, 1.7998 bits con T=1 y 2.2888 bits con T=2. La entropía crece monótonamente con T: al enfriar (T=0.5) ya cae a 0.9147 bits, en camino del 0 de la delta (greedy), y al calentar (T=2) sube a 2.2888 bits, acercándose al máximo log2(6) de la uniforme. Las distribuciones acompañan ese movimiento: con T=0.5 t0 concentra 0.8231 y t5 cae a 3.7367e-05 (concentración), mientras que con T=2 las probabilidades se reparten (t5 sube a 0.0294 y t0 baja a 0.3583), es decir, aplanamiento hacia la uniforme.